This source-preserving adaptation of Notebook16 confirmation retains its tested power, seal, exposure-ledger and statistical rules. Notebook17 screening seeds are explicitly excluded. No test pixels are read during import or registration. Runtime: definitions only; actual confirmation is gated and has its own immutable 12-hour cap. Checkpoint/resume: source-bound phase protocol, latest two/full-state model checkpoints and converged classifier receipts.

# Notebook17 confirmation definitions
Purpose: predeclare one candidate, calculate training-seed count, seal final weights before opening the official evaluation subset, and analyze paired results. Inputs: screening effects, pinned metadata and completed source-bound model records. Outputs: immutable registration/seal and paired inference. Runtime: under one minute for metadata/statistics; feature scoring is handled by the converged evaluator. Checkpoints: referenced final states under checkpoints/<run_id>/. Exact resume: same registration, sources, seal and subset; invalid hashes raise an error.

In [ ]:
from pathlib import Path
import json, os
if "__file__" not in globals():
    __file__=str(Path(os.environ.get("FMM_JEPA_ROOT","/home/benjamin/workspace/ahr_go/fmm_jeppa"))/"experiments/17_fixed_kernel/confirmation.ipynb")
_CONFIRMATION_ROOT=Path(__file__).resolve().parents[2]
_boot={"__file__":str(_CONFIRMATION_ROOT/"fmm_jepa/bootstrap.ipynb")}
for _cell in json.loads((_CONFIRMATION_ROOT/"fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if _cell["cell_type"]=="code" and "ahr-export"in _cell.get("metadata",{}).get("tags",[]):
        exec(compile("".join(_cell["source"]),"bootstrap#ahr-export","exec"),_boot)
_boot["install_notebook_package"](_CONFIRMATION_ROOT)


In [ ]:
"""Frozen single-candidate confirmation; official test stays closed until a valid seal."""
from pathlib import Path
import copy, hashlib, json, math, random, statistics, time
import numpy as np
from scipy import stats
from fmm_jepa.training import atomic_json,digest_file
from fmm_jepa.food101_training import run_writer

def hash_object(value):return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
def read(path):return json.loads(Path(path).read_text())
def immutable(path,value):
    path=Path(path);value=json.loads(json.dumps(value,sort_keys=True,allow_nan=False))
    if path.exists():
        if read(path)!=value:raise ValueError('Frozen confirmation record changed: '+str(path))
    else:atomic_json(path,value)
    return value
def power_plan(pilot_deltas,min_effect_pp=.5,alpha=.05,target_power=.8,variance_inflation=1.5,min_pairs=8,max_pairs=32):
    values=np.asarray(pilot_deltas,dtype=float)
    if len(values)<3 or not np.isfinite(values).all():raise ValueError('At least three finite paired pilot effects required')
    if not(0<alpha<1 and 0<target_power<1 and min_effect_pp>0 and variance_inflation>=1 and 2<=min_pairs<=max_pairs):raise ValueError('Invalid power specification')
    observed=float(np.std(values,ddof=1));planned=max(.25,observed*variance_inflation);rows=[];chosen=None
    for n in range(min_pairs,max_pairs+1):
        critical=stats.t.ppf(1-alpha/2,n-1);noncentral=math.sqrt(n)*min_effect_pp/planned
        # Noncentral-t symmetry avoids SciPy's NaN in an extreme left-tail CDF.
        power=float(stats.nct.sf(critical,n-1,-noncentral)+stats.nct.sf(critical,n-1,noncentral))
        if not math.isfinite(power) or not 0.0<=power<=1.0:raise ValueError('Nonfinite or invalid noncentral-t power; feasibility is UNKNOWN')
        rows.append({'pairs':n,'assumed_power':power})
        if chosen is None and power>=target_power:chosen=n
    return {'status':'FEASIBLE_SEED_COUNT'if chosen is not None else'UNKNOWN_POWER_FEASIBILITY','pairs':chosen,
      'pilot_pairs':len(values),'pilot_sd_pp':observed,'planned_sd_pp':planned,'variance_inflation':variance_inflation,
      'minimum_detectable_effect_pp':min_effect_pp,'alpha':alpha,'target_power':target_power,'minimum_pairs':min_pairs,'maximum_pairs':max_pairs,
      'method':'Two-sided paired-t power from a noncentral t distribution; training seed is the inferential unit. Pilot SD is inflated conservatively, with a 0.25 pp floor.',
      'assumptions':'Pilot variance is uncertain; inference assumes an appropriate paired-effect distribution. Projected power is not realized significance.', 'curve':rows}
def select_closed_ids(bundle,per_class=50,seed=20262017):
    ids=list(bundle['heldout_ids']);groups={};forbidden=set(bundle['train_ids'])|set(bundle['probe_train_ids'])|set(bundle['dev_ids'])
    if forbidden&set(ids):raise ValueError('Heldout membership leaked into training/development')
    for sid in ids:
        if not sid.startswith('food101:test:'):raise ValueError('Only original official-test IDs may enter confirmation')
        groups.setdefault(int(bundle['dataset'].index[sid]['label']),[]).append(sid)
    if len(groups)!=101 or any(len(v)<per_class for v in groups.values()):raise ValueError('Complete balanced official-test class coverage unavailable')
    rng=random.Random(seed);selected=[]
    for label in sorted(groups):selected.extend(sorted(rng.sample(sorted(groups[label]),per_class)))
    return {'ids':selected,'membership_sha256':hash_object(selected),'selection_seed':seed,'per_class':per_class,
      'images':len(selected),'classes':101,'dataset_fingerprint':bundle['dataset'].fingerprint,
      'metadata_only':True,'predictive_pixels_read':0,'status':'CLOSED'}
def normalized_probe_config(root,plan):
    import types
    from fmm_jepa.bootstrap import TaggedNotebookLoader
    path=Path(root)/'experiments/15_vit_geometry/evaluation.ipynb';module=types.ModuleType('fmm17_registered_probe');module.__file__=str(path);TaggedNotebookLoader(path).exec_module(module)
    return module.probe_configuration(plan.get('probe_convex'))
def verify_sources(root,sources):
    root=Path(root).resolve()
    for relative,expected in sources.items():
        path=(root/relative).resolve()
        if not path.is_relative_to(root)or path.suffix!='.ipynb':raise ValueError('Invalidregistereddefinitionpath')
        source='\n'.join(''.join(c['source'])for c in json.loads(path.read_text())['cells']if c['cell_type']=='code'and'ahr-export'in c.get('metadata',{}).get('tags',[]))
        if hashlib.sha256(source.encode()).hexdigest()!=expected:raise ValueError('Registereddefinitionchanged:'+relative)
    return True
def prior_confirmation_seeds(root,artifact=None):
    target=(Path(artifact)/'confirmation_registration.json').resolve()if artifact is not None else None;used=set()
    for parent in ('notebook_16','notebook_17'):
        for path in (Path(root)/'artifacts'/parent).rglob('confirmation_registration.json'):
            if target is not None and path.resolve()==target:continue
            used.update(map(int,read(path).get('seeds',[])))
    return used

def fresh_confirmation_seeds(root,artifact,n,start=20262001):
    used=prior_confirmation_seeds(root,artifact);seeds=[];candidate=int(start)
    while len(seeds)<int(n):
        if candidate not in used:seeds.append(candidate)
        candidate+=1
    return seeds
def register_confirmation(root,artifact,plan,pilot_deltas,bundle,configuration=None):
    root,artifact=Path(root),Path(artifact);cfg=configuration or {};power=power_plan(pilot_deltas,
      min_effect_pp=float(cfg.get('minimum_effect_pp',.5)),alpha=float(cfg.get('alpha',.05)),target_power=float(cfg.get('target_power',.8)),
      variance_inflation=float(cfg.get('variance_inflation',1.5)),min_pairs=int(cfg.get('min_pairs',8)),max_pairs=int(cfg.get('max_pairs',32)))
    if power['pairs'] is None:raise TimeoutError('Requested confirmation power cannot be planned within registered seed cap')
    seeds=list(plan.get('registered_seeds',plan.get('seeds',[])))
    if len(seeds)!=power['pairs']or len(set(seeds))!=len(seeds):raise ValueError('Confirmation seeds must match the frozen power-derived count')
    used=set(range(20261401,20261407))|set(range(20261501,20261509))|set(range(20261601,20261605))|set(range(20261701,20261705))
    used|=set(range(20261901,20261905))|prior_confirmation_seeds(root,artifact)
    if used&set(seeds):raise ValueError('Pilot, selection or prior confirmation seeds cannot enter fresh confirmation')
    registry_path=root/'data/food101/final_evaluation_registry.json'
    registry=read(registry_path)if registry_path.exists()else{'runs':{}}
    old=read(artifact/'confirmation_registration.json')if(artifact/'confirmation_registration.json').exists()else None
    if old is not None:heldout=old['closed_holdout']
    else:
        exposed={sid for entry in registry['runs'].values()for sid in entry['ids']}
        available={**bundle,'heldout_ids':[sid for sid in bundle['heldout_ids']if sid not in exposed]}
        heldout=select_closed_ids(available,int(cfg.get('heldout_per_class',50)),int(cfg.get('heldout_seed',20262017)))
    if plan.get('variants')!=['vit','optional_geometry']:raise ValueError('Exactly one preselected candidate and ViT required')
    reg={'schema':'notebook17-confirmation-registration/v1','run_id':plan['run_id'],'created_unix':time.time(),
      'plan_hash':hash_object(plan),'source_hashes':plan['source_hashes'],'seeds':seeds,'variants':plan['variants'],
      'power':power,'closed_holdout':heldout,'probe_configuration':normalized_probe_config(root,plan),'probe_training_ids_sha256':hash_object(list(bundle['probe_train_ids'])),'primary_metric':'Balanced official-test top-1 accuracy after equal active wall time in full JEPA training; converged linear probe with fixed L2 penalty.',
      'inferential_unit':'Independent paired training seed; image count is not seed count','primary_family':['fixed_kernel_candidate_minus_vit'],
      'alpha':power['alpha'],'minimum_effect_pp':power['minimum_detectable_effect_pp'],'comparison_budget':plan.get('active_wall_seconds_per_arm'),
      'success_rule':'All pairs and endpoints complete and finite, probes converged, mean gain at least the registered minimum, and the paired 95% t interval entirely above zero.',
      'secondary':'Exact two-sided sign test for seed-effect direction; kNN is descriptive. No selection of the best seed, epoch or holdout result.',
      'checkpoint_selection':'Last completed update within the frozen active wall allowance; no development-based checkpoint choice',
      'opened_status':'CLOSED','test_pixels_read':0,'limitations':['Pilot variance is uncertain; projected power is not a guarantee.','The 5,050-image subset is balanced and reserved from predictive selection; claims are conditional on this population.']}
    path=artifact/'confirmation_registration.json'
    if path.exists():
        old=read(path);reg['created_unix']=old['created_unix']
    immutable(path,reg);return reg
def verify_gate_evidence(root,gate_receipts):
    root=Path(root).resolve()
    for gate in gate_receipts:
        record=gate.get('evidence')
        if not isinstance(record,dict)or not record.get('path')or not record.get('sha256'):raise ValueError('Gate evidence file and checksum required')
        path=Path(record['path']).resolve()
        if not path.is_relative_to(root/'artifacts')or not path.is_file()or digest_file(path)!=record['sha256']:raise ValueError('Gate evidence file changed or outside scientific artifacts')
    return True

def seal_final_evaluation(root,artifact,registration,models,gate_receipts):
    root,artifact=Path(root),Path(artifact)
    if registration!=read(artifact/'confirmation_registration.json'):raise ValueError('Registered final protocol changed')
    verify_sources(root,registration['source_hashes'])
    if registration.get('opened_status')!='CLOSED':raise ValueError('Initial frozen registration mustremainCLOSED')
    required={(seed,variant)for seed in registration['seeds']for variant in registration['variants']}
    actual={(int(row['seed']),row['variant'])for row in models}
    if actual!=required or len(models)!=len(required):raise ValueError('Every fresh paired model required before final opening')
    for gate in gate_receipts:
        if gate.get('status')!='PASS'or not gate.get('passed',False)or gate.get('fixture_source'):raise ValueError('Actualpreflight/quality/compression/runtimegate did not pass')
    if not {'quality','operator','full_jepa_cost'}.issubset({g.get('role')for g in gate_receipts}):raise ValueError('Quality, operator+gradient, and measuredfullJEPAcost gate receiptsrequired')
    verify_gate_evidence(root,gate_receipts)
    frozen=[]
    for row in models:
        if row.get('status')!='COMPLETE'or not row.get('fair_wall_budget_verified',False):raise ValueError('Incomplete orunequal-wall trainingcannotopenofficialevaluation')
        rec=row['source_encoder'];path=Path(rec['path']).resolve()
        if not path.is_relative_to(root/'checkpoints'/registration['run_id']):raise ValueError('Finalweightsmustbelongtofreshregisteredrun')
        if digest_file(path)!=rec['sha256']:raise ValueError('Finalcheckpointchecksum mismatch')
        frozen.append(copy.deepcopy(row))
    seal={'schema':'notebook17-final-evaluation-seal/v1','registration_path':str(artifact/'confirmation_registration.json'),
      'registration_sha256':digest_file(artifact/'confirmation_registration.json'),'registration_hash':hash_object(registration),
      'plan_hash':registration['plan_hash'],'source_hashes':registration['source_hashes'],'heldout_membership_sha256':registration['closed_holdout']['membership_sha256'],
      'models':frozen,'gate_receipts':gate_receipts,'all_registered_models_frozen':True,'state':'AUTHORIZED_FINAL_EVALUATION',
      'selection_after_opening':'FORBIDDEN','opened_unix':time.time()}
    p=artifact/'final_evaluation_seal.json'
    if p.exists():seal['opened_unix']=read(p)['opened_unix']
    # Reserve this subset durably before first possible prediction. An interrupted
    # opening remains exposed conservatively, and cannot be reused as fresh data.
    registry_path=root/'data/food101/final_evaluation_registry.json'
    with run_writer(root/'data/food101/final_evaluation_registry_writer'):
        registry=read(registry_path)if registry_path.exists()else{'schema':'food101-final-exposure-registry/v1','runs':{}}
        reg_sha=seal['registration_sha256'];ids=registration['closed_holdout']['ids'];key=registration['run_id']
        for other,entry in registry['runs'].items():
            if other!=key and set(entry['ids'])&set(ids):raise ValueError('Finalsubsetwasalreadyreserved/opened byanotherexperiment')
        if key in registry['runs']:
            entry=registry['runs'][key]
            if entry['registration_sha256']!=reg_sha or entry['ids']!=ids:raise ValueError('Existingfinalexposureidentitychanged')
            seal['opened_unix']=entry['opened_unix']
        else:
            registry['runs'][key]={'registration_sha256':reg_sha,'ids':ids,'opened_unix':seal['opened_unix'],'policy':'Reserved conservatively before prediction; these images are not fresh for later studies.'}
            atomic_json(registry_path,registry)
        return immutable(p,seal)
def verify_seal(root,artifact,seal,requested_ids,model_record=None):
    root,artifact=Path(root),Path(artifact);p=artifact/'final_evaluation_seal.json'
    if not p.exists()or read(p)!=seal or seal.get('state')!='AUTHORIZED_FINAL_EVALUATION':raise PermissionError('Avalidfrozenfinal-evaluationsealisrequired')
    regpath=Path(seal['registration_path'])
    if regpath!=artifact/'confirmation_registration.json'or digest_file(regpath)!=seal['registration_sha256']:raise ValueError('Finalregistrationchecksum mismatch')
    reg=read(regpath);verify_sources(root,reg['source_hashes'])
    verify_gate_evidence(root,seal['gate_receipts'])
    if hash_object(reg)!=seal['registration_hash']or hash_object(list(requested_ids))!=seal['heldout_membership_sha256']:raise ValueError('Finalmembership/protocol changed')
    if model_record is not None:
        rec=model_record.get('source_encoder',model_record)
        if not any(row['source_encoder']['path']==rec['path']and row['source_encoder']['sha256']==rec['sha256']for row in seal['models']):raise ValueError('Unsealedmodel')
        if digest_file(rec['path'])!=rec['sha256']:raise ValueError('Frozencheckpointchanged')
    return reg
def analyze_confirmation(registration,rows):
    seeds=registration['seeds'];variants=registration['variants'];effects=[];paired=[]
    for seed in seeds:
        group={r['variant']:r for r in rows if int(r['seed'])==seed}
        if set(group)!=set(variants)or sum(int(r['seed'])==seed for r in rows)!=len(variants):return {'status':'UNKNOWN','reason':'Missingor duplicatepaired endpoints'}
        for r in group.values():
            if r.get('status')!='COMPLETE'or not r.get('converged')or not r.get('numerical_valid',False)or r.get('fixture_source'):return {'status':'UNKNOWN','reason':'Nonconverged/invalid/non-scientific finalendpoint'}
            if not isinstance(r.get('accuracy_percent'),(int,float))or not math.isfinite(float(r['accuracy_percent']))or not 0<=float(r['accuracy_percent'])<=100:return {'status':'UNKNOWN','reason':'Invalid final accuracy'}
            if r.get('heldout_membership_sha256')!=registration['closed_holdout']['membership_sha256']:raise ValueError('Finalpredictionmembershipdiffers')
        delta=float(group['optional_geometry']['accuracy_percent']-group['vit']['accuracy_percent']);effects.append(delta)
        paired.append({'seed':seed,'vit_percent':group['vit']['accuracy_percent'],'candidate_percent':group['optional_geometry']['accuracy_percent'],'delta_pp':delta})
    if len(rows)!=2*len(seeds):raise ValueError('Unregisteredendpointincluded')
    values=np.asarray(effects);n=len(values);mean=float(values.mean());sd=float(values.std(ddof=1));sem=sd/math.sqrt(n)
    if sd==0:
        ci=None;p=None;method='UNDEFINED_ZERO_VARIANCE_PAIRED_T; exactdirectional sensitivityreported separately'
    else:
        ci=list(map(float,stats.t.interval(1-registration['alpha'],n-1,loc=mean,scale=sem)));p=float(stats.ttest_1samp(values,0).pvalue);method='Registeredtwo-sidedpairedt'
    positives=int((values>0).sum());negatives=int((values<0).sum());nonzero=positives+negatives
    sign=float(stats.binomtest(positives,nonzero,.5,alternative='two-sided').pvalue)if nonzero else 1.
    significant=ci is not None and ci[0]>0 and p is not None and p<registration['alpha'];meaningful=mean>=registration['minimum_effect_pp']
    return {'status':'COMPLETE','decision':'UNKNOWN_DEGENERATE_VARIANCE'if ci is None else'SUPPORTS_REGISTERED_ACCURACY_GAIN'if significant and meaningful else'NO_CONFIRMED_MEANINGFUL_ACCURACY_GAIN',
      'n_training_pairs':n,'mean_delta_pp':mean,'raw_ci95_pp':ci,'paired_t_two_sided_p':p,'method':method,
      'minimum_effect_pp':registration['minimum_effect_pp'],'positive_pairs':positives,'negative_pairs':negatives,'ties':n-nonzero,
      'exact_sign_test_two_sided_p':sign,'paired_results':paired,'primary_family_size':1,
      'limitations':['Inference is conditional on the paired-effect distribution and fixed official subset.','Exact sign sensitivity is reported separately; images are not independent training replicates.','Nonsignificance does not establish equivalence or improvement.']}
def run_validation(root,artifact=None):
    artifact=Path(artifact or Path(root)/'artifacts/notebook_17_validation/confirmation');artifact.mkdir(parents=True,exist_ok=True)
    a=power_plan([-.2,.2,.3,.4]);assert a['pairs']>=8
    b=power_plan([-5.,5.,-4.,4.]);assert b['status']=='UNKNOWN_POWER_FEASIBILITY'
    reg={'seeds':list(range(8)),'variants':['vit','optional_geometry'],'alpha':.05,'minimum_effect_pp':.5,'closed_holdout':{'membership_sha256':'fixture'}}
    rows=[]
    for seed in reg['seeds']:
        for variant in reg['variants']:
            rows.append({'seed':seed,'variant':variant,'accuracy_percent':10.+(.8+.03*seed if variant=='optional_geometry'else 0.),
              'status':'COMPLETE','converged':True,'numerical_valid':True,'heldout_membership_sha256':'fixture'})
    good=analyze_confirmation(reg,rows);assert good['decision']=='SUPPORTS_REGISTERED_ACCURACY_GAIN'
    missing=analyze_confirmation(reg,rows[:-1]);assert missing['status']=='UNKNOWN'
    changed=copy.deepcopy(rows);changed[0]['heldout_membership_sha256']='wrong'
    try:analyze_confirmation(reg,changed)
    except ValueError:pass
    else:raise AssertionError('Changedholdoutaccepted')
    result={'status':'PASS','fixture_source':'SOFTWARE_FIXTURE; NOT_CONFIRMATION_RESULTS','checks':5,'power_example_pairs':a['pairs'],'source_export_sha256':export_hash()}
    atomic_json(artifact/'receipt.json',result);return result
def export_hash():
    p=Path(__file__);return hashlib.sha256('\n'.join(''.join(c['source'])for c in json.loads(p.read_text())['cells']if c['cell_type']=='code'and'ahr-export'in c.get('metadata',{}).get('tags',[])).encode()).hexdigest()

def prepare_registration(root,artifact,handoff,bench,budget_remaining,configuration=None):
    """Persist power and active-wall allocation before fresh confirmation training."""
    root,artifact=Path(root),Path(artifact);cfg=configuration or {};path=artifact/'confirmation_proposal.json'
    if handoff.get('status')!='COMPLETE'or not handoff.get('quality_pass')or handoff.get('fixture_source'):return {'status':'NO_GO_QUALITY','reason':'No valid actual17candidate passed its registered quality screen'}
    for name,key in [('candidate_plan','plan_sha256'),('training_result','training_sha256'),('evaluation_result','evaluation_sha256')]:
        if digest_file(handoff[name])!=handoff[key]:raise ValueError('Handoffsourcechecksum changed:'+name)
    identity={'handoff_hash':hash_object(handoff),'benchmark_hash':hash_object(bench),'configuration':cfg,'confirmation_export_sha256':export_hash()}
    if path.exists():
        saved=read(path)
        if saved['identity']!=identity:raise ValueError('Frozenconfirmationproposal changed')
        return saved
    power=power_plan(handoff['screening']['paired_seed_deltas_pp'],min_effect_pp=float(cfg.get('minimum_effect_pp',.5)),
      alpha=float(cfg.get('alpha',.05)),target_power=float(cfg.get('target_power',.8)),variance_inflation=float(cfg.get('variance_inflation',1.5)),
      min_pairs=int(cfg.get('min_pairs',8)),max_pairs=int(cfg.get('max_pairs',32)))
    if power['pairs']is None:return {'status':'UNKNOWN_POWER_FEASIBILITY','power':power,'identity':identity}
    n=power['pairs'];gpu_workers=int(cfg.get('gpu_workers',2))
    if gpu_workers not in(1,2):raise ValueError('One or two available GPU workers must be frozen')
    seconds=min(1800,int(.65*float(budget_remaining)*gpu_workers/(2*n)))
    if seconds<600:return {'status':'UNKNOWN_BUDGET_FEASIBILITY','reason':'Powerderivedpairsdo not fit600sminimumactivewall perarm plusreservedreadout allowance','power':power,'identity':identity}
    seeds=fresh_confirmation_seeds(root,artifact,n);proposal={'status':'READY_FOR_FROZEN_CONFIRMATION','identity':identity,'power':power,'seeds':seeds,'pairs':n,
      'active_wall_seconds_per_arm':seconds,'gpu_workers':gpu_workers,'schedule_updates':4000,'minimum_effect_pp':power['minimum_detectable_effect_pp'],
      'alpha':power['alpha'],'target_power':power['target_power'],'evaluation_reserve_seconds':max(3600,int(.3*float(budget_remaining))),
      'heldout_per_class':50,'heldout_seed':20262017,'official_test':'CLOSED','runtime_estimate_seconds':2*n*seconds/gpu_workers,
      'runtime_estimate_scope':'Registered GPU worker count; roughcomputeprojectionexcludesoperatorpreflight andfeature/probe overhead; absolute12hguard remainsauthoritative',
      'schedule_policy':'Same4000-successful-update scheduler horizon; maystop at differentupdates under sameactivewall. Hittingupdatecap early invalidatesfairness.'}
    return immutable(path,proposal)
def bind_sealed_evaluation(root,model_builder,source_hashes):
    """Bind our validator immediately before either official-data verification."""
    from fmm_jepa.bootstrap import TaggedNotebookLoader
    import types
    path=Path(root)/'experiments/15_vit_geometry/evaluation.ipynb'
    module=types.ModuleType('fmm17_sealed_evaluation');module.__file__=str(path);TaggedNotebookLoader(path).exec_module(module)
    module.bind_model_builder(model_builder,source_hashes)
    # The frozen15 evaluator defaults to16's verifier. Bind only this isolated
    # instance to17 so gate evidence SHA checks recur before first test pixels.
    module._seal_verifier=lambda requested_root:verify_seal
    return module

def seal_and_score(root,artifact,plan,bundle,training,evaluation,deadline):
    """Open only the preselected official subset after all paired weights and gates freeze."""
    root,artifact=Path(root),Path(artifact);registration=read(artifact/'confirmation_registration.json')
    if registration['plan_hash']!=hash_object(plan):raise ValueError('Finalplan differsfrom preregistration')
    if training.get('status')!='COMPLETE'or not training.get('fair_wall_budget_verified',False):return {'status':'UNKNOWN','reason':'Incompleteor unverifiedequal-active-walltraining'}
    arms=training.get('arms',{});models=[]
    for arm in plan['arms']:
        if arm['id']not in arms:raise ValueError('Missingfreshtrainedarm')
        saved=arms[arm['id']];source=saved.get('source_encoder',saved.get('latest_checkpoint'))
        if not source:raise ValueError('Finaltrainedarmhasnofrozenencoderrecord')
        models.append({'seed':arm['seed'],'variant':arm['variant'],'arm_id':arm['id'],'status':'COMPLETE',
          'fair_wall_budget_verified':bool(saved.get('fair_wall_budget_verified',training['fair_wall_budget_verified'])),'source_encoder':source})
    receipts=training.get('gate_receipts',plan.get('gate_receipts',[]))
    seal=seal_final_evaluation(root,artifact,registration,models,receipts)
    if callable(evaluation):model_builder=evaluation
    elif isinstance(evaluation,dict)and callable(evaluation.get('model_builder')):model_builder=evaluation['model_builder']
    else:raise ValueError('Final readout requires the frozen17 model builder')
    module=bind_sealed_evaluation(root,model_builder,plan['source_hashes'])
    rows=[];heldout=registration['closed_holdout']['ids']
    with run_writer(artifact/'final_evaluation_writer'):
        for arm,model in zip(plan['arms'],models):
            if time.time()>=deadline:return {'status':'UNKNOWN','reason':'Finalreadoutcutoff','completed_endpoints':len(rows),'official_test':'OPENED_FOR_FROZEN_PROTOCOL_ONLY'}
            score_config=copy.deepcopy(arm['config']);score_config['probe_convex']=registration['probe_configuration']
            candidates=evaluation.get('rows',[])if isinstance(evaluation,dict)else[]
            prior=[r for r in candidates if int(r['seed'])==int(arm['seed'])and r['variant']==arm['variant']]
            if len(prior)>1:raise ValueError('Duplicate converged training probe for final model')
            probe_record=prior[0]['probe_receipt']if prior and prior[0].get('converged')else None
            record=module.evaluate_model_on_ids(root,artifact,score_config,model['source_encoder'],bundle['probe_train_ids'],heldout,bundle['dataset'],
              training_probe_record=probe_record,deadline_unix=deadline,seal_record=seal)
            record.update(seed=arm['seed'],variant=arm['variant'],heldout_membership_sha256=registration['closed_holdout']['membership_sha256'])
            rows.append(record);atomic_json(artifact/'final_evaluation_progress.json',{'status':'RUNNING','completed_endpoints':len(rows),'total_endpoints':len(plan['arms']),'rows':rows})
            if record.get('status')!='COMPLETE'or not record.get('converged'):return {'status':'UNKNOWN','reason':'Final endpoint incomplete or classifier not converged','completed_endpoints':len(rows),'rows':rows}
        result=analyze_confirmation(registration,rows);result.update(official_test='EVALUATED_ON_FROZEN5050_SUBSET',final_evaluation_seal_sha256=digest_file(artifact/'final_evaluation_seal.json'),rows=rows)
        atomic_json(artifact/'confirmation_results.json',result);return result
